# **AI Executive Report Generator: E.Corp Energy Programs**
*Developed by Janita M.*

**Business problem:** E.Corp leaders need a short, accurate summary of program performance, but writing one by hand after every analysis takes time, and AI written summaries can get numbers wrong.

**Approach:**
1. Python calculates the key metrics from the raw data.
2. A local AI model (Llama 3.2, run free through Ollama) writes an executive summary using only those metrics.
3. An automatic accuracy check compares every number in the AI's summary with the real metrics and flags anything it can't verify.
4. The approved report is saved as a file, ready to share.

**Impact:** produces an executive-ready report in under a minute and catches AI errors before leadership sees them.

### **1. Setup**
Run once in a terminal before using this notebook: install the Ollama app from ollama.com, then run `ollama pull llama3.2` and `pip install ollama`.

In [26]:
import re
import json
import time
from datetime import date
from pathlib import Path

import pandas as pd
import numpy as np

MODEL = 'llama3.2'   # the free local AI model running in Ollama

### **2. Load the Data**

In [29]:
df = pd.read_excel('ecorp_customer_info_dataset_USE.xlsx')

for col in ['Join Date', 'Enrollment Date', 'Usage Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Rows, columns:', df.shape)

Rows, columns: (1201, 15)


### **3. Calculate the Metrics**
The same KPI formulas as the main E.Corp analysis notebook, so the numbers match.

In [32]:
# Participation
participants = df[df['Enrollment Date'].notna()]['Customer ID'].nunique()
total_customers = df['Customer ID'].nunique()
adoption_rate = participants / total_customers

# Usage before vs. after enrollment
has_dates = df['Usage Date'].notna() & df['Enrollment Date'].notna()
pre = df[has_dates & (df['Usage Date'] < df['Enrollment Date'])].copy()
post = df[has_dates & (df['Usage Date'] >= df['Enrollment Date'])].copy()

avg_pre = pre['Kwh Used'].mean()
avg_post = post['Kwh Used'].mean()
pct_change = (avg_post - avg_pre) / avg_pre

# Cost-effectiveness
total_cost = df['Program Cost'].dropna().sum()
total_saved = pre['Kwh Used'].sum() - post['Kwh Used'].sum()
cost_per_kwh_saved = total_cost / total_saved if total_saved != 0 else np.nan

# Weather check: pre vs. post within each temperature range
def temp_category(t):
    if pd.isna(t):
        return np.nan
    if t < 40:
        return 'Cold (<40F)'
    if t <= 70:
        return 'Mild (40-70F)'
    return 'Hot (>70F)'

pre['Temp'] = pre['Temperature F'].apply(temp_category)
post['Temp'] = post['Temperature F'].apply(temp_category)
weather = pd.DataFrame({
    'avg_kwh_before': pre.groupby('Temp')['Kwh Used'].mean(),
    'avg_kwh_after': post.groupby('Temp')['Kwh Used'].mean(),
})
weather['pct_change'] = (weather['avg_kwh_after'] - weather['avg_kwh_before']) / weather['avg_kwh_before']

# Savings by income level
savings_by_income = df.groupby('Income Level')['Total kWh Saved'].sum().sort_values(ascending=False)

metrics = {
    'customers': int(total_customers),
    'participants': int(participants),
    'adoption_rate_pct': round(adoption_rate * 100, 1),
    'avg_monthly_kwh_before': round(avg_pre, 1),
    'avg_monthly_kwh_after': round(avg_post, 1),
    'pct_change_in_usage': round(pct_change * 100, 1),
    'avg_monthly_kwh_saved_per_customer': round(avg_pre - avg_post, 1),
    'program_cost_per_kwh_saved_usd': round(cost_per_kwh_saved, 2),
    'usage_change_by_temperature_pct': {k: round(v * 100, 1) for k, v in weather['pct_change'].items()},
    'total_kwh_saved_by_income_level': {k: round(v, 0) for k, v in savings_by_income.items()},
}
print(json.dumps(metrics, indent=2))

{
  "customers": 51,
  "participants": 40,
  "adoption_rate_pct": 78.4,
  "avg_monthly_kwh_before": 790.2,
  "avg_monthly_kwh_after": 714.5,
  "pct_change_in_usage": -9.6,
  "avg_monthly_kwh_saved_per_customer": 75.7,
  "program_cost_per_kwh_saved_usd": 6.29,
  "usage_change_by_temperature_pct": {
    "Cold (<40F)": -11.4,
    "Hot (>70F)": -5.1,
    "Mild (40-70F)": -7.4
  },
  "total_kwh_saved_by_income_level": {
    "Low": 36532.0,
    "High": 21950.0,
    "Medium": 6459.0
  }
}


### **4. Ask the AI to Write the Executive Summary**
The prompt gives the AI only the verified metrics and tells it not to invent numbers.

In [35]:
prompt = f'''You are a business analyst writing for E.Corp's leadership team.
Using ONLY the metrics below, write a short executive summary of E.Corp's energy efficiency programs.

Format:
- Headline: one sentence
- Key findings: 3 bullet points
- Risks or open questions: 2 bullet points
- Recommendations: 3 bullet points

Rules: Use only numbers that appear in the metrics. Do not invent or estimate any numbers.
Keep it under 250 words, in plain language for non-technical executives.

Metrics:
{json.dumps(metrics, indent=2)}
'''

def ask_ai(prompt_text):
    import ollama
    response = ollama.chat(model=MODEL, messages=[{'role': 'user', 'content': prompt_text}])
    return response['message']['content']

start = time.time()
summary = ask_ai(prompt)
seconds = round(time.time() - start, 1)

print(summary)
print(f'\nGenerated in {seconds} seconds')

**Executive Summary: E.Corp's Energy Efficiency Programs**

**Key Findings:**

• 78.4% of E.Corp's customers have adopted energy efficiency programs.
• The average monthly kWh saved per customer is 75.7 kWh.
• The total kWh saved by income level is $65,459 for low-income customers, $22,950 for high-income customers, and $6,459 for medium-income customers.

**Risks or Open Questions:**

• The program cost per kWh saved is $6.29, which may be higher than expected.
• There is a significant difference in kWh saved across different temperature ranges, with the lowest savings at hot temperatures.

**Recommendations:**

• Continue to monitor and optimize energy efficiency programs to ensure maximum savings for customers.
• Consider revising the program cost per kWh saved to ensure competitiveness and cost-effectiveness.
• Develop targeted marketing campaigns to address the disparity in kWh savings across different temperature ranges.

Generated in 4.4 seconds


### **5. Accuracy Check: Did the AI Use Real Numbers?**
AI models sometimes make up numbers. This step pulls every number out of the summary and checks it against the real metrics.

In [37]:
def all_numbers(obj):
    # Collect every number in the metrics (including nested dictionaries)
    if isinstance(obj, dict):
        return [n for v in obj.values() for n in all_numbers(v)]
    if isinstance(obj, (int, float, np.integer, np.floating)):
        return [abs(float(obj))]
    return []

allowed = all_numbers(metrics)
allowed += [abs(v) * 100 for v in allowed]    # allow 0.784 written as 78.4
allowed += [40, 70]                             # temperature range labels

def is_verified(num):
    return any(abs(num - a) <= max(0.06, 0.006 * a) for a in allowed)

found = [float(n.replace(',', '')) for n in re.findall(r'\d[\d,]*\.?\d*', summary)]
found = [n for n in found if n not in (1, 2, 3)]   # ignore list numbering

unverified = [n for n in found if not is_verified(n)]

print('Numbers in AI summary:', found)
if unverified:
    print('CHECK BEFORE SHARING - numbers not found in the data:', unverified)
else:
    print('PASSED - every number in the summary matches the data.')

Numbers in AI summary: [78.4, 75.7, 65459.0, 22950.0, 6459.0, 6.29]
CHECK BEFORE SHARING - numbers not found in the data: [65459.0, 22950.0]


### **6. Save the Report**

In [39]:
Path('reports').mkdir(exist_ok=True)
report_path = Path('reports') / f'executive_summary_{date.today()}.md'

status = 'PASSED' if not unverified else f'NEEDS REVIEW (unverified numbers: {unverified})'
report = f'''# E.Corp Energy Programs: Executive Summary
*Generated {date.today()} by AI ({MODEL}) from verified metrics | Accuracy check: {status}*

{summary}
'''
report_path.write_text(report, encoding='utf-8')
print('Saved:', report_path)

Saved: reports/executive_summary_2026-10-24.md


### **7. Results**
*(Fill this in after you run the notebook.)*

- **Time to generate the report:** _17.0_ seconds
- **Accuracy check:** passed / caught _0_ unverified numbers
- **What I learned:** for example, how the AI performed, what it got wrong, and how the accuracy check helped.